# page_devide.wav 마커 소리 기준 MP3 자동 분할

**원리 — 템플릿 매칭 (정규화 상호상관, NCC)**

1. 마커 소리 파일(`page_devide.wav`)과 각 MP3를 같은 샘플레이트로 맞춘 뒤
   STFT 로그 크기 스펙트럼으로 변환합니다.
2. 템플릿 길이의 창을 MP3 위에서 1프레임씩 슬라이드하며 창 내용과 템플릿의
   **코사인 유사도(NCC)** 곡선을 계산합니다.
3. 곡선에서 `match_thr` 이상의 피크 = "마커 소리가 나는 위치"로 판정합니다.
4. 마커 구간(± `guard` 여백)을 제외한 나머지 구간을 세그먼트로 잘라
   트랙별 폴더에 MP3로 저장합니다.

- 비프음처럼 **완전히 같은 소리**든, 책장 소리처럼 **비슷한 소리**든
  템플릿 파일만 바꿔 그대로 재사용할 수 있습니다.
- 트랙마다 샘플레이트가 달라도 자동으로 맞춰 처리합니다.

In [ ]:
import os, csv, glob
import numpy as np
import soundfile as sf
from scipy import signal

BASE_DIR   = 'D:/SEMCOWork/Session26_mp3'
MP3_FOLDER = f'{BASE_DIR}/Disney Fun to Read 1'        # 분할 대상 MP3 폴더
TPL_PATH   = f'{BASE_DIR}/page_devide.wav'             # 마커 소리 파일 (작업 폴더에 넣어주세요)
OUT_ROOT   = f'{BASE_DIR}/split_by_marker'             # 저장 위치 (트랙별 하위 폴더 생성)
IDX_CSV    = f'{BASE_DIR}/split_by_marker_index.csv'   # 세그먼트 인덱스
MK_CSV     = f'{BASE_DIR}/split_by_marker_markers.csv' # 검출된 마커 목록

CFG = dict(
    n_fft=1024,      # STFT 창 길이(샘플) ≈ 23ms
    hop=256,         # STFT 프레임 간격(샘플) ≈ 5.8ms
    match_thr=0.7,   # 마커 판정 임계값(유사도 0~1): 오탐이 많으면 ↑, 놓치면 ↓
    min_gap=1.0,     # 마커 사이 최소 간격(초)
    guard=0.05,      # 마커 앞뒤로 잘라내는 여백(초)
    min_seg=0.2,     # 이보다 짧은 세그먼트는 버림(초)
)
print('설정 완료 | CFG =', CFG)

In [ ]:
def load_mono(path, target_sr=None):
    """오디오를 모노로 읽고, target_sr 이 지정되면 리샘플링해서 반환"""
    data, sr = sf.read(path, always_2d=True)
    x = data.mean(axis=1)
    if target_sr and sr != target_sr:
        g = np.gcd(int(sr), int(target_sr))
        x = signal.resample_poly(x, target_sr // g, sr // g)
        sr = target_sr
    return x.astype(np.float64), sr

def log_mag_stft(x, sr, n_fft, hop):
    """로그 크기 스펙트럼 (주파수 × 시간프레임) 반환 — 절대 크기 스케일은 NCC에서 상쇄됨"""
    _, _, Z = signal.stft(x, fs=sr, nperseg=n_fft, noverlap=n_fft - hop,
                          padded=False, boundary=None)
    return np.log1p(1000 * np.abs(Z))

def ncc_curve(S, T):
    """슬라이딩 윈도우 코사인 유사도 곡선
    corr[t] = <S[:, t:t+K], T> / (||S[:, t:t+K]|| · ||T||)  — 1에 가까울수록 유사"""
    F, Ta = S.shape
    K = T.shape[1]
    if Ta < K:
        return np.zeros(0)
    corr = np.zeros(Ta - K + 1)
    for f in range(F):  # 주파수 밴드별 상관을 누적 (FFT 상관)
        corr += signal.fftconvolve(S[f], T[f, ::-1], mode='valid')
    e = np.convolve((S ** 2).sum(axis=0), np.ones(K), mode='valid')
    et = (T ** 2).sum()
    # 거의 무음인 창(e ≈ 0)은 0으로 마스킹 — 무음에서의 허수 피크 방지
    e_floor = (e.max() * 1e-3) if len(e) else 1.0
    return np.where(e > e_floor, corr / np.sqrt(np.maximum(e, 1e-12) * et), 0.0)

def detect_markers(x, sr, tpl, tpl_sr, cfg):
    """오디오에서 템플릿과 유사한 구간 검출
    반환: (마커 목록 [(시작s, 끝s, 유사도)], NCC 곡선, 초당 프레임 수)"""
    if tpl_sr != sr:  # 템플릿을 트랙 샘플레이트에 맞춤
        g = np.gcd(int(tpl_sr), int(sr))
        tpl = signal.resample_poly(tpl, sr // g, tpl_sr // g)
    S = log_mag_stft(x, sr, cfg['n_fft'], cfg['hop'])
    T = log_mag_stft(tpl, sr, cfg['n_fft'], cfg['hop'])
    ncc = ncc_curve(S, T)
    fps = sr / cfg['hop']
    K = T.shape[1]
    if len(ncc) == 0:
        return [], ncc, fps
    idx, _ = signal.find_peaks(ncc, height=cfg['match_thr'],
                               distance=max(1, int(cfg['min_gap'] * fps)))
    markers = [(t / fps, (t + K) / fps, float(ncc[t])) for t in idx]
    return markers, ncc, fps

def compute_segments(markers, total, guard, min_seg):
    """마커 구간(±guard)을 제외한 남은 구간 [(시작, 끝), ...]"""
    segs, cur = [], 0.0
    for m0, m1, _ in markers:
        if m0 - guard - cur >= min_seg:
            segs.append((cur, m0 - guard))
        cur = max(cur, m1 + guard)
    if total - cur >= min_seg:
        segs.append((cur, total))
    return segs

print('함수 준비 완료')

In [ ]:
if not os.path.exists(TPL_PATH):
    raise FileNotFoundError(
        f'마커 소리 파일이 없습니다: {TPL_PATH}\n'
        '기준이 될 소리 파일을 page_devide.wav 라는 이름으로 작업 폴더에 넣어주세요.')

tpl, tpl_sr = load_mono(TPL_PATH)
print(f'템플릿: {os.path.basename(TPL_PATH)}')
print(f'  길이 {len(tpl) / tpl_sr:.2f}s | 샘플레이트 {tpl_sr}Hz')

mp3s = sorted(glob.glob(os.path.join(MP3_FOLDER, '*.mp3')))
print(f'대상 MP3: {len(mp3s)}개')

In [ ]:
# 트랙 1 미리보기 — 검출 위치와 유사도 분포를 확인하고 임계값 조정에 참고
x, sr = load_mono(mp3s[0])
markers, ncc, fps = detect_markers(x, sr, tpl, tpl_sr, CFG)
print(f'트랙1 ({len(x) / sr:.1f}s) | 마커 {len(markers)}개 검출 (임계값 {CFG["match_thr"]})')
if len(ncc):
    print(f'NCC 분포: max={ncc.max():.3f} / p99={np.percentile(ncc, 99):.3f} / median={np.median(ncc):.3f}')
    pk, _ = signal.find_peaks(ncc, distance=max(1, int(CFG['min_gap'] * fps)))
    top = pk[np.argsort(ncc[pk])[::-1][:8]]
    print('NCC 상위 피크 8개 (임계값 무관 — 튜닝 참고용):')
    for t in sorted(top):
        print(f'   {t / fps:7.2f}s  유사도 {ncc[t]:.3f}')
print()
print('검출된 마커 (임계값 이상):')
for m0, m1, sc in markers[:15]:
    print(f'   {m0:7.2f}s ~ {m1:6.2f}s  (유사도 {sc:.3f})')
if len(markers) > 15:
    print(f'   ... 외 {len(markers) - 15}개')

In [ ]:
# 전체 트랙 실행 — 마커 검출 → 마커 구간 제외 세그먼트 저장
os.makedirs(OUT_ROOT, exist_ok=True)
mk_rows, all_rows = [], []
for i, path in enumerate(mp3s, 1):
    x, sr = load_mono(path)
    total = len(x) / sr
    markers, ncc, fps = detect_markers(x, sr, tpl, tpl_sr, CFG)
    for m0, m1, sc in markers:
        mk_rows.append(dict(track=i, marker_start=f'{m0:.2f}',
                            marker_end=f'{m1:.2f}', score=f'{sc:.3f}'))
    segs = compute_segments(markers, total, CFG['guard'], CFG['min_seg'])
    if not markers:
        print(f'  track {i:2d} | 마커 0개 — 트랙 전체가 세그먼트 1개로 저장됨')
    tdir = os.path.join(OUT_ROOT, f'track{i:02d}')
    os.makedirs(tdir, exist_ok=True)
    for j, (a, b) in enumerate(segs, 1):
        fname = f'track{i:02d}_seg{j:02d}_{a:.2f}-{b:.2f}s.mp3'
        sf.write(os.path.join(tdir, fname), x[int(a * sr):int(b * sr)], sr, format='MP3')
        all_rows.append(dict(track=i, start=f'{a:.2f}', end=f'{b:.2f}',
                             dur=f'{b - a:.2f}', markers=len(markers),
                             file=fname, folder=f'track{i:02d}'))
    print(f'  track {i:2d} | 마커 {len(markers):2d}개 → 세그먼트 {len(segs):2d}개')

with open(MK_CSV, 'w', newline='', encoding='utf-8-sig') as fp:
    w = csv.DictWriter(fp, fieldnames=['track', 'marker_start', 'marker_end', 'score'])
    w.writeheader()
    w.writerows(mk_rows)
with open(IDX_CSV, 'w', newline='', encoding='utf-8-sig') as fp:
    w = csv.DictWriter(fp, fieldnames=list(all_rows[0].keys()))
    w.writeheader()
    w.writerows(all_rows)
print(f'\n세그먼트 {len(all_rows)}개 | 마커 {len(mk_rows)}개')
print(f'저장: {OUT_ROOT}')
print(f'인덱스: {IDX_CSV}')
print(f'마커 목록: {MK_CSV}')

In [ ]:
# 검증 — 1) 세그먼트가 마커 구간과 겹치지 않는지  2) 저장 파일 길이 일치
mk_by_track = {}
for r in mk_rows:
    mk_by_track.setdefault(int(r['track']), []).append(
        (float(r['marker_start']), float(r['marker_end'])))

bad = []
for r in all_rows:
    tr, a, b = int(r['track']), float(r['start']), float(r['end'])
    for m0, m1 in mk_by_track.get(tr, []):
        if a < m1 + CFG['guard'] - 0.01 and b > m0 - CFG['guard'] + 0.01:
            bad.append((r['file'], m0, m1))
            break
print(f'[구간 검증] 마커와 겹치는 세그먼트: {len(bad)}개 →', 'PASS' if not bad else 'FAIL')
for f, m0, m1 in bad[:5]:
    print(f'   {f} (마커 {m0:.2f}~{m1:.2f}s)')

import random
random.seed(0)
sample = random.sample(all_rows, min(5, len(all_rows)))
ok = 0
for r in sample:
    y, ysr = sf.read(os.path.join(OUT_ROOT, r['folder'], r['file']))
    if abs(len(y) / ysr - float(r['dur'])) <= 0.02:
        ok += 1
print(f'[파일 검증] 무작위 {len(sample)}개 재읽기 길이 일치: {ok}/{len(sample)} →',
      'PASS' if ok == len(sample) else 'FAIL')

## 사용 방법

1. 기준 소리 파일을 `page_devide.wav` 이름으로 작업 폴더(`D:\SEMCOWork\Session26_mp3`)에 넣기
2. 위 셀들을 위에서부터 실행 → `split_by_marker\track01\` … 에 분할 MP3 저장
3. 결과 확인 후 기준을 바꾸려면 템플릿 파일만 교체하고 다시 실행

## 파라미터 튜닝 (설정 셀의 CFG)

| 파라미터 | 의미 | 조정 기준 |
|---|---|---|
| `match_thr` | 마커 판정 유사도 임계값 | 오탐(없는 소리 검출)이 많으면 ↑, 놓치면 ↓ — 미리보기 셀의 NCC 분포·상위 피크 참고 |
| `min_gap` | 마커 사이 최소 간격(초) | 마커가 촘촘하면 ↓ |
| `guard` | 마커 앞뒤 잘라내는 여백(초) | 마커 소리가 세그먼트에 새면 ↑ |
| `min_seg` | 최소 세그먼트 길이(초) | 너무 짧은 조각 방지 |

- 템플릿은 **마커 소리 1회 분량**(예: 비프음 1번, 책장 소리 1번)으로 준비하세요.
- 마커 구간 자체는 결과에서 제외됩니다(소리가 새지 않도록 ±guard 여백 포함).